# Autoresearch Experiment Analysis (Latency)

Analysis of autonomous inference-latency optimization results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 6 columns: commit, latency_ms, correctness, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["latency_ms"] = pd.to_numeric(df["latency_ms"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["correctness"] = df["correctness"].astype(str).str.strip().str.lower()
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    lat = row["latency_ms"]
    desc = row["description"]
    print(f"  #{i:3d}  latency={lat:.3f}ms  mem={row['memory_gb']:.1f}GB  {desc}")

## Latency Over Time

Track how the best (kept) `latency_ms` evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy()
valid = valid.reset_index(drop=True)

baseline_latency = valid.loc[0, "latency_ms"]

# Only plot points at or below baseline + small margin (the interesting region)
below = valid[valid["latency_ms"] <= baseline_latency * 1.02]

# Plot discarded as faint background dots
disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["latency_ms"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["latency_ms"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line
kept_mask = valid["status"] == "KEEP"
kept_idx = valid.index[kept_mask]
kept_latency = valid.loc[kept_mask, "latency_ms"]
running_min = kept_latency.cummin()
ax.step(kept_idx, running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, lat in zip(kept_idx, kept_latency):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (idx, lat),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Latency (ms, lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below best to just above baseline
best_latency = kept_latency.min()
margin = (baseline_latency - best_latency) * 0.15
ax.set_ylim(best_latency - margin, baseline_latency + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_latency = df.iloc[0]["latency_ms"]
best_latency = kept["latency_ms"].min()
best_row = kept.loc[kept["latency_ms"].idxmin()]

print(f"Baseline latency: {baseline_latency:.3f} ms")
print(f"Best latency:     {best_latency:.3f} ms")
print(f"Total speedup:    {baseline_latency - best_latency:.3f} ms ({(baseline_latency - best_latency) / baseline_latency * 100:.2f}%)")
print(f"Best experiment:  {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative progress:")
kept_sorted = kept.reset_index()
for i, (_, row) in enumerate(kept_sorted.iterrows()):
    desc = str(row["description"]).strip()
    print(f"  Experiment #{row['index']:3d}: latency={row['latency_ms']:.3f}ms  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's latency
# (since experiments are cumulative -- each one builds on the last kept state)
kept = df[df["status"] == "KEEP"].copy()
kept["prev_latency"] = kept["latency_ms"].shift(1)
kept["delta"] = kept["prev_latency"] - kept["latency_ms"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta (ms)':>11}  {'Latency (ms)':>13}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+11.3f}  {row['latency_ms']:13.3f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+11.3f}  {'':>13}  TOTAL speedup over baseline")